##### Dataset

In [ ]:
"""Generate half-circles and moons datasets with 1000 and 10000 samples"""

from sklearn.datasets import make_moons
import numpy as np

#make moon datasets
X_1k_moon, y_1k_moon = make_moons(n_samples=1000, noise=0.1, random_state=42)
X_10k_moon, y_10k_moon = make_moons(n_samples=10000, noise=0.1, random_state=42)

#make half-circles datasets
X_1k_half = X_1k_moon.copy()
X_10k_half = X_10k_moon.copy()
#shift the second class down by 1.5 to create half-circles
X_1k_half[y_1k_moon == 1, 1] -= 1.5
X_10k_half[y_10k_moon == 1, 1] -= 1.5

y_1k_half = y_1k_moon.copy()
y_10k_half = y_10k_moon.copy()

#test the shapes of the datasets
print("Shapes of the Moon datasets:")
print(X_1k_moon.shape, y_1k_moon.shape)
print(X_10k_moon.shape, y_10k_moon.shape)
print("\n" + "-"*40 + "\n")
print("Shapes of the Half-Circle datasets:")
print(X_1k_half.shape, y_1k_half.shape)
print(X_10k_half.shape, y_10k_half.shape)

##### Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))

# 1K moon dataset
plt.scatter(
    X_1k_moon[y_1k_moon == 0, 0], X_1k_moon[y_1k_moon == 0, 1],
    color='blue',
    edgecolor='k',
    s=20,
    label='Class 0'
)
plt.scatter(
    X_1k_moon[y_1k_moon == 1, 0], X_1k_moon[y_1k_moon == 1, 1],
    color='red',
    edgecolor='k',
    s=20,
    label='Class 1'
)
plt.title("Make Moons Dataset n=1000")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

# 10k moon dataset
plt.scatter(
    X_10k_moon[y_10k_moon == 0, 0], X_10k_moon[y_10k_moon == 0, 1],
    color='blue',
    edgecolor='k',
    s=20,
    label='Class 0'
)
plt.scatter(
    X_10k_moon[y_10k_moon == 1, 0], X_10k_moon[y_10k_moon == 1, 1],
    color='red',
    edgecolor='k',
    s=20,
    label='Class 1'
)
plt.title("Make Moons Dataset n=10000")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()


# 1K half-circle dataset
plt.scatter(
    X_1k_half[y_1k_half == 0, 0], X_1k_half[y_1k_half == 0, 1],
    color='blue',
    edgecolor='k',
    s=20,
    label='Class 0'
)
plt.scatter(
    X_1k_half[y_1k_half == 1, 0], X_1k_half[y_1k_half == 1, 1],
    color='red',
    edgecolor='k',
    s=20,
    label='Class 1'
)
plt.title("Make Half-Circle Dataset n=1000")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

# 10k half-circle dataset
plt.scatter(
    X_10k_half[y_10k_half == 0, 0], X_10k_half[y_10k_half == 0, 1],
    color='blue',
    edgecolor='k',
    s=20,
    label='Class 0'
)
plt.scatter(
    X_10k_half[y_10k_half == 1, 0], X_10k_half[y_10k_half == 1, 1],
    color='red',
    edgecolor='k',
    s=20,
    label='Class 1'
)
plt.title("Make Half-Circle Dataset n=10000")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()



##### Standardize Dataset

In [ ]:
def standardize(X):
    mu = X.mean(axis=0)
    std = X.std(axis=0)
    return mu, std

def standardize_transform(X, mu, std):
    return (X - mu) / std

##### K-fold Cross Validation

In [ ]:
"""Implement general k-fold cross validation (use 5 folds)"""
# This function generates train and test indices for k-fold cross-validation.
# random_state is used to ensure reproducibility when shuffling the indices.

def kfold_indices(n, k=5, shuffle=True, random_state=42):
    # generate random numbers with the given random state for reproducibility
    rng = np.random.default_rng(random_state)

    # split the indices into k folds
    idx = np.arange(n)
    if shuffle: rng.shuffle(idx)
    folds = np.array_split(idx, k)

    # 1 fold = test, remaining k-1 folds = train
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.hstack([folds[j] for j in range(k) if j != i])
        yield train_idx, test_idx

##### Metrics

In [ ]:
def metrics(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (TP + TN) / len(y_true)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

##### Simple MLP
MLP (Multi-Layer Perceptron) is a type of feedforward neural network that learns non-linear relationships between inputs and outputs.
Input layer  →  Hidden layer(s)  →  Output layer

In [ ]:
class SimpleMLP:
    def __init__(self, n_hidden=32, lr=0.05, epochs=180, batch_size=128, l2=1e-4, seed=42):
        # n_hidden: number of neurons in the hidden layer; 
        self.n_hidden=n_hidden
        self.lr=lr
        self.epochs=epochs
        self.batch_size=batch_size
        #batch_size: control how many samples are used to compute one gradient update during training. It is beneficial for speed and memory efficiency, especially for large datasets.
        self.l2=l2
        self.rng=np.random.default_rng(seed)

    def _init(self, d):
        # initialize weights and biases based on input dimension d.
        """rng.normal() generates random numbers from a normal distribution:
           The first argument is the mean of the distribution.
           The second argument is the standard deviation, which is scaled 
           by the input dimension d or n_hidden to help with stability during training."""
        
        """ Input -> Hidden Layer """
        self.W1 = self.rng.normal(0, 1/np.sqrt(d), size=(d, self.n_hidden))
        self.b1 = np.zeros(self.n_hidden)

        """ Hidden Layer -> Output Layer """
        self.W2 = self.rng.normal(0, 1/np.sqrt(self.n_hidden), size=(self.n_hidden, 1))
        self.b2 = np.zeros(1)

    """ Activation function: tanh for hidden layer, sigmoid for output layer.
        Loss: binary cross-entropy + L2 regularization. """
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1, 1)
        
        n,d=X.shape
        self._init(d)
        for _ in range(self.epochs):
            # randomly shuffle the data at the beginning of each epoch for better training
            perm=self.rng.permutation(n)

            for start in range(0,n,self.batch_size):
                idx=perm[start:start+self.batch_size]
                xb=X[idx]; yb=y[idx]

                """ Forward pass: compute the activations and predicted probabilities."""
                # hidden layer
                z1=xb@self.W1 + self.b1
                a1=np.tanh(z1) # activation function for hidden layer: tanh

                # output layer
                z2=a1@self.W2 + self.b2
                p=1/(1+np.exp(-z2)) # activation function for output layer: sigmoid, p is the predicted probability of class 1
                
                """ Backpropagation: compute gradients and update weights and biases."""
                # From chatGPT, use prompt "backpropagation for binary classification with sigmoid output and binary cross-entropy loss" to 
                # derive the gradients, which results in the following formulas:
                """ δ2 = 1/m1 (p - y)
                    dW2 = a1^T δ2 + λ W2   
                    db2 = sum(δ2)
                    δ1 = δ2 W2^T * (1 - a1^2) 
                    dW1 = xb^T δ1 + λ W1
                    db1 = sum(δ1)
                """

                # --- output layer grdients ---
                # derivate of loss w.r.t. z2, averaged over the batch size
                # sigmoid + binary cross-entropy loss has a nice property that the gradient simplifies to (p-y).
                dz2=(p-yb)/len(idx)
                # derivative of loss w.r.t. W2, includes L2 regularization term self.l2*self.W2
                dW2=a1.T@dz2 + self.l2*self.W2
                # derivative of loss w.r.t. b2
                db2=dz2.sum(axis=0)

                # --- backprop into hidden layer ---
                # derivative of loss w.r.t. z1
                da1=dz2@self.W2.T
                dz1=da1*(1-a1**2)
                # derivative of loss w.r.t. W1, includes L2 regularization term self.l2*self.W1
                dW1=xb.T@dz1 + self.l2*self.W1
                # derivative of loss w.r.t. b1
                db1=dz1.sum(axis=0)

                # --- parameter update ---
                self.W2 -= self.lr*dW2
                self.b2 -= self.lr*db2
                self.W1 -= self.lr*dW1
                self.b1 -= self.lr*db1
        return self

    def predict(self, X):
        z1 = X@self.W1 + self.b1
        a1=np.tanh(z1)
        z2 = (a1@self.W2 + self.b2).ravel()
        p=1/(1+np.exp(-z2))
        return (p>=0.5).astype(int)

##### SVM with kernels
Support Vector Machine (SVM) tries to find a decision boundary that separates classes with the maximum margin.
Kernel SVM extends it so the boundary can be nonlinear.


**Linear Kernal** Measure similarity using a simple dot product; best for linearly separable data with fast computation.
-> K(x,z)=x⋅z

**Polynomial kernal** Captures more complex relationships by raising feature interactions to a power, allowing curved decision boudnaries.
-> K(x,z)=(γx⋅z+r)^d

**RBF kernal** Maps data into infinite dimensions and works well when class boundaries are higly non-linear and irregular.
-> K(x,z)=exp(−γ||x−z||^2) = exp(-γ * ( ||x||^2 + ||z||^2 - 2x^T z))

**Sigmoid Kernel** Behaves like a neural netweok activation function. 
-> K(x,z)=tanh(γx⋅z+r)

x and z are two feature vectors. 

#### Dual Objective: max(​λ)​ ∑​λi​−1/2 ∑(i,j) ​λi​λj​yi​yj​K(xi​,xj​)
                subject to: 0≤λ(i)​≤C, ∑​λi​yi​=0
    prediction: f(x)=i∑​λi​yi​K(xi​,x)+b


In [ ]:
class KernelSVM:
    def __init__(self, C=1.0, kernel='rbf', sigma=0.1, alpha=1, c=0, degree=2, max_kernel_samples=2500, random_state=42):
        # C：regularization parameter that controls the trade-off between maximizing the margin and minimizing classification errors. 
        # Smaller C creates a wider margin but allows more misclassifications; Bigger C creates a narrower margin but will classify more training data points correctly.
        
        # kernel: type of kernel function to use - 'linear', 'rbf', 'poly', and 'sigmoid'
        
        # sigma: parameter for RBF kernel, controls the width of the Gaussian function. Smaller sigma creates a narrower kernel, while larger sigma creates a wider kernel.
        
        # alpha: scaling parameter for polynomial and sigmoid kernels.
        
        # c: bias term for polynomial and sigmoid kernels.
        
        # degree: degree of the polynomial kernel function.

        self.C = C
        self.max_kernel_samples = max_kernel_samples
        self.random_state = random_state
        self.kernel_name = kernel
        if kernel == 'linear':
            self.kernel = self.linear_kernel
        elif kernel == 'rbf':
            self.kernel = self.rbf_kernel
            self.sigma = sigma
        elif kernel == 'poly':
            self.kernel = self.poly_kernel
            self.alpha = alpha
            self.c = c
            self.degree = degree
        elif kernel == 'sigmoid':
            self.kernel = self.sigmoid_kernel
            self.alpha = alpha
            self.c = c
        else:
            raise ValueError("Unsupported kernel type. Choose from 'linear', 'rbf', 'poly', or 'sigmoid'.")

        self.X = None
        self.Z = None
        self.lmbda = None # lagrangeian multipliers
        self.b = 0 # bias term
        self.ones = None
        self.w = None

    def linear_kernel(self, X, Z):
        return X @ Z.T
    
    def rbf_kernel(self, X, Z):
        X2=np.sum(X**2,axis=1)[:,None]
        Z2=np.sum(Z**2,axis=1)[None,:]
        return np.exp(-self.sigma*(X2 + Z2 - 2 * X @ Z.T))
    
    def poly_kernel(self, X, Z): 
        return (self.alpha * X @ Z.T + self.c) ** self.degree
    
    def sigmoid_kernel(self, X, Z):
        return np.tanh(self.alpha * X @ Z.T + self.c)
    
    def fit(self, X, y, lr = 1e-3, epochs = 200):
        y = np.where(y == 0, -1, 1) # convert labels to -1 and 1 for SVM

        n = X.shape[0]

        if self.kernel_name == 'linear':
            # Primal linear SVM update avoids O(n^2) kernel matrix construction.
            self.w = np.zeros(X.shape[1])
            self.b = 0.0
            reg = 1.0 / max(self.C, 1e-12)

            for _ in range(epochs):
                margin = y * (X @ self.w + self.b)
                mask = margin < 1.0
                if np.any(mask):
                    grad_w = reg * self.w - np.mean(y[mask, None] * X[mask], axis=0)
                    grad_b = -np.mean(y[mask])
                else:
                    grad_w = reg * self.w
                    grad_b = 0.0

                self.w -= lr * grad_w
                self.b -= lr * grad_b
            return
        if self.max_kernel_samples is not None and n > self.max_kernel_samples:
            # Use a deterministic subset so kernel methods remain tractable on large folds.
            rng = np.random.default_rng(self.random_state)
            pos = np.where(y == 1)[0]
            neg = np.where(y == -1)[0]
            n_sub = self.max_kernel_samples
            n_pos = min(len(pos), n_sub // 2)
            n_neg = min(len(neg), n_sub - n_pos)
            if n_pos + n_neg < n_sub:
                extra = n_sub - (n_pos + n_neg)
                n_pos = min(len(pos), n_pos + extra)
                n_neg = min(len(neg), n_sub - n_pos)
            sub_idx = np.concatenate([
                rng.choice(pos, size=n_pos, replace=False) if n_pos > 0 else np.array([], dtype=int),
                rng.choice(neg, size=n_neg, replace=False) if n_neg > 0 else np.array([], dtype=int),
            ])
            rng.shuffle(sub_idx)
            X = X[sub_idx]
            y = y[sub_idx]
            n = X.shape[0]

        self.X = X
        self.y = y

        self.lmbda = np.zeros(n) # initialize lagrangeian multipliers to 0
        self.b = 0 # initialize bias term to 0
        self.ones = np.ones(n) # vector of ones for bias term

        y_outer = np.outer(y, y) 
        K = self.kernel(X, X) 
        y_iy_jk_ij = y_outer * K 

        yy = y @ y  # precompute y^T y for projection step

        for _ in range(epochs):
            gradient = self.ones - y_iy_jk_ij @ self.lmbda
            self.lmbda += lr * gradient
            
            for _proj in range(2):
                self.lmbda -= y * (y @ self.lmbda) / (yy)   
                self.lmbda = np.clip(self.lmbda, 0.0, self.C)

            idx = np.where((self.lmbda > 1e-8) & (self.lmbda < self.C - 1e-8))[0]
            if len(idx) > 0:
                b_i = y[idx] - (self.lmbda * y) @ K[:, idx]
                self.b = float(np.mean(b_i))

    def predict(self, X):
        if self.kernel_name == 'linear' and self.w is not None:
            decision = X @ self.w + self.b
            return (decision >= 0).astype(int)

        K = self.kernel(self.X, X)
        decision = (self.lmbda * self.y) @ K + self.b
        return (decision >= 0).astype(int)

##### K-fold Cross Validation Results

In [ ]:

def evaluate_model_cv(model_factory, X, y, k=5, seed=42):
    fold_rows = []
    times = []
    for fold, (tr, te) in enumerate(kfold_indices(len(X), k=k, shuffle=True, random_state=seed), start=1):
        Xtr, ytr = X[tr], y[tr]
        Xte, yte = X[te], y[te]

        # standardize using TRAIN datasets only
        mu, sd = standardize(Xtr)
        Xtr_Standardize = standardize_transform(Xtr, mu, sd)
        Xte_Standardize = standardize_transform(Xte, mu, sd)

        model = model_factory()

        model.fit(Xtr_Standardize, ytr)

        ytr_pred = model.predict(Xtr_Standardize)
        yte_pred = model.predict(Xte_Standardize)

        m_tr = metrics(ytr, ytr_pred)
        m_te = metrics(yte, yte_pred)

        #print(f"Fold {fold}: Train Acc={m_tr['accuracy']:.3f}, Train F1={m_tr['f1']:.3f}, Test Acc={m_te['accuracy']:.3f}, Test F1={m_te['f1']:.3f}")

        fold_rows.append({
            "fold": fold,
            "train_accuracy": m_tr["accuracy"], "train_precision": m_tr["precision"], "train_recall": m_tr["recall"], "train_f1": m_tr["f1"],
            "test_accuracy":  m_te["accuracy"], "test_precision":  m_te["precision"], "test_recall":  m_te["recall"], "test_f1":  m_te["f1"],
        })
    mean_metrics = {
        "train_accuracy": np.mean([r["train_accuracy"] for r in fold_rows]),
        "train_f1":       np.mean([r["train_f1"] for r in fold_rows]),
        "test_accuracy":  np.mean([r["test_accuracy"] for r in fold_rows]),
        "test_f1":        np.mean([r["test_f1"] for r in fold_rows]),
    }
    return mean_metrics

def print_results_table(title, results_dict):
    # results_dict: name -> mean_metrics
    headers = ["Model", "Train Acc", "Train F1", "Test Acc", "Test F1"]
    print("\n" + title)
    print("-"*len(title))
    print("{:<18s} {:>9s} {:>9s} {:>9s} {:>9s}".format(*headers))
    for name, m in results_dict.items():
        print("{:<18s} {:>9.3f} {:>9.3f} {:>9.3f} {:>9.3f}".format(
            name,
            m["train_accuracy"], m["train_f1"],
            m["test_accuracy"],  m["test_f1"]
        ))

##### Run Experiment

In [33]:
model_factories = {
        "MLP": lambda: SimpleMLP(n_hidden=32, lr=0.05, epochs=180, batch_size=128, l2=1e-4, seed=42),
        "SVM-linear":  lambda: KernelSVM(kernel="linear", C=1.0, sigma=0.1, alpha=1, c=0, degree=2, max_kernel_samples=None),
        "SVM-poly":    lambda: KernelSVM(kernel="poly", C=1.0, sigma=0.1, alpha=0.1, c=0, degree=3, max_kernel_samples=2500, random_state=42),
        "SVM-rbf":     lambda: KernelSVM(kernel="rbf", C=10, sigma=0.5, alpha=1, c=0, max_kernel_samples=2500, random_state=42),
        "SVM-sigmoid": lambda: KernelSVM(kernel="sigmoid", C=1.0, sigma=0.1, alpha=0.01, c=0.5, max_kernel_samples=2500, random_state=42),
    }

results = {}
print("5-Fold Cross Validation Results")
for name, factory in model_factories.items():
    #print(f"\nEvaluating {name} on Moons with 1000 samples...")
    results[name] = evaluate_model_cv(factory, X_1k_moon, y_1k_moon, k=5, seed=42)
print_results_table("Moons with 1000 samples", results)

for name, factory in model_factories.items():
    #print(f"\nEvaluating {name} on Moons with 10000 samples...")
    results[name] = evaluate_model_cv(factory, X_10k_moon, y_10k_moon, k=5, seed=42)
print_results_table("Moons with 10000 samples", results)

for name, factory in model_factories.items():
    results[name] = evaluate_model_cv(factory, X_1k_half, y_1k_half, k=5, seed=42)
print_results_table("Half-circles with 1000 samples", results)

for name, factory in model_factories.items():
    results[name] = evaluate_model_cv(factory, X_10k_half, y_10k_half, k=5, seed=42)
print_results_table("Half-circles with 10000 samples", results)




5-Fold Cross Validation Results

Moons with 1000 samples
-----------------------
Model              Train Acc  Train F1  Test Acc   Test F1
MLP                    0.942     0.942     0.941     0.940
SVM-linear             0.861     0.860     0.860     0.858
SVM-poly               0.860     0.860     0.845     0.848
SVM-rbf                0.998     0.998     0.998     0.998
SVM-sigmoid            0.865     0.865     0.864     0.862

Moons with 10000 samples
------------------------
Model              Train Acc  Train F1  Test Acc   Test F1
MLP                    0.998     0.998     0.998     0.998
SVM-linear             0.862     0.862     0.862     0.862
SVM-poly               0.863     0.864     0.864     0.865
SVM-rbf                0.998     0.998     0.998     0.998
SVM-sigmoid            0.878     0.878     0.878     0.878

Half-circles with 1000 samples
------------------------------
Model              Train Acc  Train F1  Test Acc   Test F1
MLP                    1.000     1.000